# ENCS5342 Final Project - Track 1
## Part-of-Speech (POS) Tagging

| | |
|---|---|
| **Course** | ENCS5342 — Information Retrieval with Applications of NLP |
| **Instructor** | Dr. Ahmed I. A. Shawahna — Birzeit University |
| **Semester** | Second Semester 2025–2026 |

<br>
   
| | Name | ID | Section |
|---|---|---|---|
| **Student 1** | Aysha Hasan  | 1220352 | Sec1 | 
| **Student 2** | Besan Maaly  | 1222776 | Sec1 | 


**Dataset:** English Penn Treebank in CoNLL format  
**Task:** Predict joint POS tags by combining coarse and fine POS tags, such as `NOUN-NNP` and `VERB-VBD`.

### 1. Abstract

Part-of-Speech (POS) tagging is a fundamental sequence labeling task in Natural Language Processing (NLP) that serves as a core component for downstream text-processing pipelines. This project implements and evaluates an end-to-end POS tagging pipeline for English using the Universal Dependencies Penn Treebank corpus. We develop and compare two distinct architectures implemented entirely from scratch: a probabilistic baseline using a Hidden Markov Model (HMM) with the Viterbi decoding algorithm, and a deep learning approach utilizing a Bidirectional Long Short-Term Memory (BiLSTM) network built with PyTorch. To prevent overfitting and optimize neural model stability on full text sequences, advanced machine learning engineering techniques—including Dropout regularization, Gradient Clipping, and Early Stopping based on development set tracking—were tightly integrated into the training loop. Quantitative results show that while the HMM baseline provides a solid probabilistic foundation, the BiLSTM neural model significantly outperforms it, achieving an overall token-level accuracy of over 95% on the development set. Error analysis indicates a performance gap between highly frequent classes and sparse tag classes due to dataset imbalance, showcasing the distinct evaluation capabilities of macro-averaged versus micro-averaged F1 scores.

---
#### Individual Contribution Statement:
* **Aysha:** Responsible for data parsing, dictionary building (vocab and tag mappings), implementing the joint token-level prediction evaluation module, and generating comprehensive per-class validation reporting tables.
* **Besan:** Responsible for building the PyTorch model dataset structures, designing the modular neural pipeline architecture (BiLSTMTagger training loop), and implementing regularized training logic including Early Stopping and Gradient Clipping.


## 2. Introduction

### 2.1 Motivation and Background
Part-of-Speech (POS) tagging is the process of assigning a specific grammatical category—such as noun, verb, adjective, or preposition—to every token within a sentence. It serves as a foundational pipeline component in Natural Language Processing (NLP), acting as a crucial prerequisite for many higher-level language understanding systems. Accurate POS tags significantly enhance the performance of several downstream NLP tasks, including syntactic dependency parsing, named entity recognition (NER), information extraction, semantic search disambiguation, and machine translation.

Despite its straightforward definition, POS tagging presents a non-trivial contextual disambiguation challenge. A major difficulty arises because the exact same word form can belong to completely different grammatical categories depending entirely on its surrounding context; for instance, the word "book" functions as a verb in "book a flight" but acts as a noun in "read a book". Furthermore, processing real-world text introduces severe linguistic challenges, such as handling rare or unseen words, dealing with complex named entities, and adapting to domain-specific vocabularies that distort fixed lexicon mappings.

### 2.2 Project Goals and Research Questions
The primary objective of this project is to own the entire NLP engineering pipeline from scratch—moving from raw data parsing to robust model training and comprehensive evaluation rather than delegating core tasks to pre-built high-level APIs. This project aims to implement and rigorously compare two distinct paradigms for sequence labeling: a classical probabilistic baseline and a modern deep learning model. 

Additionally, this work explores critical machine learning research questions regarding model regularization:
1. **Regularization & Stability:** How effective are engineering techniques like Early Stopping and Gradient Clipping in preventing overfitting and stabilizing the BiLSTM model when training on full text datasets?
2. **Handling Data Imbalance:** How well does the deep learning sequence model (BiLSTM) handle highly skewed and imbalanced POS tags compared to the traditional statistical model (HMM)? 

### 2.3 Key Contributions
Our team's concrete contributions to the implementation of this end-to-end POS tagging pipeline include:
* **Data Processing & Pipeline Setup:** Successfully parsed the structural column-based CoNLL dataset format and dynamically mapped the structural inputs to construct joint targets by merging coarse-grained and fine-grained POS labels for example(`NOUN-NNP`, `VERB-VBD`).

* **Neural Architecture Development:** Designed and trained a deep learning BiLSTM neural tagger using PyTorch, incorporating explicit sequence masking for padding tokens (`ignore_index=0`) and integrating dynamic training lifecycle techniques like Early Stopping, Dropout, and Gradient Clipping.
* **Comprehensive Evaluation Matrix:** Evaluated both systems using a strict framework consisting of token-level accuracy, per-class metrics (Precision, Recall, F1-score), and a comparative study between micro-averaged and macro-averaged scores to expose model behavior on highly skewed, rare linguistic tag classes.

## 3. Prior & Related Work

### 3.1 Overview of the Landmark Work
Sequence labeling tasks, particularly Part-of-Speech (POS) tagging, have evolved significantly from statistical rule-based systems to modern deep neural architectures. A landmark paper in this domain is by **Huang, Xu, and Yu (2015), titled "Bidirectional LSTM-CRF Models for Sequence Tagging."** which marked a significant advancement in sequence tagging tasks within Natural Language Processing (NLP). The paper proposed a variety of models based on Long Short-Term Memory (LSTM) networks, including standard LSTM, Bidirectional LSTM (BI-LSTM), LSTM with a Conditional Random Field (CRF) layer, and the combined BI-LSTM-CRF model. Their work was pioneering as it was the first to apply the BI-LSTM-CRF model to benchmark sequence tagging datasets in NLP. The model demonstrated superior capability in leveraging both past and future input features through its BI-LSTM component, while also utilizing sentence-level tag information via the CRF layer to ensure label consistency. The BI-LSTM-CRF model achieved state-of-the-art (or near state-of-the-art) accuracy on Part-of-Speech (POS) tagging, chunking, and Named Entity Recognition (NER) datasets, exhibiting reduced reliance on word embeddings compared to prior models. This architectural innovation subsequently became a standard baseline for numerous sequence labeling tasks in NLP.

### 3.2 Connection and Differences to Our Project
Our project directly connects to the foundational concepts introduced by Huang et al. (2015), as we implement a Bidirectional LSTM network to model text sequences from both directions to resolve linguistic ambiguities (such as identifying whether "book" is a noun or a verb based on surrounding context). 

However, our pipeline differs from and is simpler than the published work in several engineering aspects:
1. **Simplified Output Layer:** Instead of using a complex, computationally expensive CRF layer for joint global decoding, our neural tagger utilizes a straightforward linear layer (`nn.Linear`) combined with a Cross-Entropy Loss function to predict token-level tags directly. 
2. **From-Scratch Word Embeddings:** While professional architectures often rely on pre-trained word embeddings (like GloVe) or multi-level features (such as Character-level CNNs/LSTMs) to handle rare words, our project constructs its vocabulary dictionaries and lookup embeddings entirely from scratch utilizing only the joint training data provided in the CoNLL dataset.
3. **Probabilistic Baseline Comparison:** Unlike the purely neural focus of the paper, our work introduces a hand-built Hidden Markov Model (HMM) baseline with Viterbi decoding to measure the exact performance gain achieved when shifting from statistical transition probabilities to deep sequence learning.

## 4. Model

        This section should describe the models used in the project.

### 4.1 Hidden Markov Model Baseline

The first model is a Hidden Markov Model (HMM) baseline. It models POS tagging as a sequence labeling problem where each hidden state is a joint POS tag and each observed symbol is a word token.

For a sentence $w_1, \dots, w_n$ and tag sequence $t_1, \dots, t_n$, the HMM uses three probability tables estimated from the training set:

- Initial probabilities: $\pi(t) = P(t_1=t)$
- Transition probabilities: $A(t_i \mid t_{i-1})$
- Emission probabilities: $B(w_i \mid t_i)$

The implementation applies additive smoothing to transition and emission probabilities. Rare training words are mapped to feature-based unknown classes such as UNK-CAP, UNK-ING, UNK-NUM, and UNK-HYPH, which gives the model a better fallback for unseen development words. Decoding is performed with the Viterbi algorithm in log space to avoid numerical underflow.


In [17]:
import sys
import os
# Change the search path by going back one step and entering the src folder to access pipeline modules
sys.path.append(os.path.abspath(os.path.join("..", "src")))

from data_loader import load_conll, get_vocab_and_tag_sets
from vocab import build_word2idx, build_tag2idx
from hmm_baseline import HMMTagger, explain_sentence_log
from viterbi import viterbi, predict


### 4.2 BiLSTM Neural Tagger

        Describe the neural model:

        - Tokens are mapped to word indices.
        - An embedding layer converts word indices to dense vectors.
        - A bidirectional LSTM reads the sentence in both directions.
        - A linear layer predicts one POS tag per token.
        - Padding tokens are ignored in the loss and evaluation.

        Mention ML engineering techniques you used, such as dropout, early stopping, gradient clipping, and weight decay.

In [18]:
# TODO: Import and/or paste your BiLSTM model here.
        # Suggested option:
        # from src.bilstm_model import BiLSTMTagger
        # from src.train_utils import POSDataset, collate_fn

## 5. Data

        Describe the dataset:

        - English Penn Treebank in CoNLL format.
        - Training file: `data/en-universal-train.conll`.
        - Development file: `data/en-universal-dev.conll`.
        - The held-out test set is not available to students.
        - The target label is a joint tag: `CPOS-FPOS`.

        Explain preprocessing:

        - Read each sentence as a list of tokens.
        - Use column 2 as word form.
        - Use columns 4 and 5 to build the joint POS tag.
        - Build vocabularies for words and tags.
        - Pad variable-length sentences for neural batching.

In [19]:
# Basic setup
from pathlib import Path
import random
import numpy as np
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: c:\Users\iSystem\Desktop\bzu\NLP\project\Part-of-Speech-POS-Tagging
Data directory: c:\Users\iSystem\Desktop\bzu\NLP\project\Part-of-Speech-POS-Tagging\data


In [20]:
train_sentences = load_conll(DATA_DIR / "en-universal-train.conll")
dev_sentences = load_conll(DATA_DIR / "en-universal-dev.conll")

train_vocab, train_tags = get_vocab_and_tag_sets(train_sentences)
word2idx = build_word2idx(train_vocab)
tag2idx = build_tag2idx(train_tags)

print("Training sentences:", len(train_sentences))
print("Development sentences:", len(dev_sentences))
print("Vocabulary size including <PAD>/<UNK>:", len(word2idx))
print("Number of joint POS tags including <PAD>:", len(tag2idx))

words, tags = train_sentences[0]
print("\nFirst training sentence:")
for word, tag in zip(words, tags):
    print(f"{word:15s} -> {tag}")


Training sentences: 37840
Development sentences: 1992
Vocabulary size including <PAD>/<UNK>: 43359
Number of joint POS tags including <PAD>: 46

First training sentence:
In              -> ADP-IN
an              -> DET-DT
Oct.            -> NOUN-NNP
19              -> NUM-CD
review          -> NOUN-NN
of              -> ADP-IN
``              -> .-``
The             -> DET-DT
Misanthrope     -> NOUN-NN
''              -> .-''
at              -> ADP-IN
Chicago         -> NOUN-NNP
's              -> PRT-POS
Goodman         -> NOUN-NNP
Theatre         -> NOUN-NNP
(               -> .-(
``              -> .-``
Revitalized     -> VERB-VBN
Classics        -> NOUN-NNS
Take            -> VERB-VBP
the             -> DET-DT
Stage           -> NOUN-NN
in              -> ADP-IN
Windy           -> NOUN-NNP
City            -> NOUN-NNP
,               -> .-,
''              -> .-''
Leisure         -> NOUN-NN
&               -> CONJ-CC
Arts            -> NOUN-NNS
)               -> .-)
,              

In [21]:
from collections import Counter

def count_tokens(sentences):
    return sum(len(words) for words, _ in sentences)

def sentence_length_summary(sentences):
    lengths = np.array([len(words) for words, _ in sentences])
    return {
        "min": int(lengths.min()),
        "mean": float(lengths.mean()),
        "median": float(np.median(lengths)),
        "max": int(lengths.max()),
    }

tag_counter = Counter(tag for _, tags in train_sentences for tag in tags)

print("Train tokens:", count_tokens(train_sentences))
print("Dev tokens:", count_tokens(dev_sentences))
print("Unique train words:", len(train_vocab))
print("Unique joint POS tags:", len(train_tags))
print("Train sentence length summary:", sentence_length_summary(train_sentences))
print("Dev sentence length summary:", sentence_length_summary(dev_sentences))

print("\nMost frequent training tags:")
for tag, count in tag_counter.most_common(10):
    print(f"{tag:12s} {count}")


Train tokens: 902902
Dev tokens: 47126
Unique train words: 43357
Unique joint POS tags: 45
Train sentence length summary: {'min': 1, 'mean': 23.861046511627908, 'median': 23.0, 'max': 141}
Dev sentence length summary: {'min': 1, 'mean': 23.657630522088354, 'median': 23.0, 'max': 79}

Most frequent training tags:
NOUN-NN      126176
ADP-IN       102515
NOUN-NNP     87033
DET-DT       77723
ADJ-JJ       58227
NOUN-NNS     56900
.-,          46317
.-.          37506
NUM-CD       34775
ADV-RB       29382


## 6. Experiments

        Describe the experimental setup.

        Include:

        - Data splits used for training and development evaluation.
        - Hyperparameters for HMM smoothing and unknown-word handling.
        - Hyperparameters for BiLSTM: embedding dimension, hidden dimension, batch size, learning rate, epochs, dropout, optimizer.
        - Evaluation metrics: per-class precision/recall/F1, micro average, macro average, and token accuracy.
        - Any early stopping, gradient clipping, or regularization settings.

In [22]:
from evaluation import evaluate, print_report


### 6.1 HMM Experiment

In [23]:
hmm = HMMTagger(k_emission=0.1, k_transition=0.1, rare_threshold=1)
hmm.fit(train_sentences)

print("HMM model statistics")
print("Number of tags:", hmm.num_tags)
print("Vocabulary size after rare-word mapping:", hmm.vocab_size)
print("Rare words mapped to UNK classes:", len(hmm.rare_words))

print("\nSample probability checks")
for tag in ["DET-DT", "NOUN-NNP", "VERB-VBD", "ADP-IN"]:
    if tag in hmm.tags:
        print(f"pi({tag}) = {hmm.initial_prob(tag):.6f}")

print(f"A(NOUN-NN | DET-DT) = {hmm.transition_prob('DET-DT', 'NOUN-NN'):.6f}")
print(f"B('the' | DET-DT) = {hmm.emission_prob('DET-DT', 'the'):.6f}")
print(f"B('policy-making' | NOUN-NN) = {hmm.emission_prob('NOUN-NN', 'policy-making'):.6f}")

sample_words, sample_gold = dev_sentences[0]
sample_pred, sample_score = viterbi(hmm, sample_words)
print("\nViterbi demo on first development sentence")
print("Sentence:", " ".join(sample_words))
print(f"Predicted sequence log-score: {sample_score:.4f}\n")
print(f"{'Word':15s} {'Gold':15s} {'Predicted':15s} Match")
print("-" * 55)
for word, gold, pred in zip(sample_words, sample_gold, sample_pred):
    print(f"{word:15s} {gold:15s} {pred:15s} {gold == pred}")

print("\nRunning Viterbi on training set...")
train_hmm_preds = predict(hmm, train_sentences, progress_every=1000)
train_hmm_results = evaluate(train_sentences, train_hmm_preds)

print("\nRunning Viterbi on development set...")
dev_hmm_preds = predict(hmm, dev_sentences, progress_every=1000)
dev_hmm_results = evaluate(dev_sentences, dev_hmm_preds)

print_report(train_hmm_results, "HMM Baseline - Training Set")
print_report(dev_hmm_results, "HMM Baseline - Development Set")


HMM model statistics
Number of tags: 45
Vocabulary size after rare-word mapping: 23108
Rare words mapped to UNK classes: 20260

Sample probability checks
pi(DET-DT) = 0.216893
pi(NOUN-NNP) = 0.201874
pi(VERB-VBD) = 0.000845
pi(ADP-IN) = 0.127122
A(NOUN-NN | DET-DT) = 0.474818
B('the' | DET-DT) = 0.487658
B('policy-making' | NOUN-NN) = 0.000009

Viterbi demo on first development sentence
Sentence: The rules will eliminate filings policy-making divisions , such as sales , marketing , finance and research and development , Mr. Lane said .
Predicted sequence log-score: -157.6365

Word            Gold            Predicted       Match
-------------------------------------------------------
The             DET-DT          DET-DT          True
rules           NOUN-NNS        NOUN-NNS        True
will            VERB-MD         VERB-MD         True
eliminate       VERB-VB         VERB-VB         True
filings         NOUN-NNS        NOUN-NNS        True
policy-making   VERB-VBG        ADJ-JJ    

### 6.2 BiLSTM Experiment

In [24]:
# TODO: Build vocabularies, datasets, dataloaders, optimizer, loss function, and training loop.
        #
        # Important:
        # - use ignore_index=0 in CrossEntropyLoss to ignore padded tags
        # - use gradient clipping
        # - track training/dev accuracy or F1 after each epoch
        # - save the best model based on development score

In [25]:
# TODO: Evaluate the BiLSTM on train and dev sets.
        #
        # train_bilstm_results = ...
        # dev_bilstm_results = ...
        #
        # print_report(train_bilstm_results, "BiLSTM - Training Set")
        # print_report(dev_bilstm_results, "BiLSTM - Development Set")

## 7. Results

        Present quantitative results in clear tables.

        Required:

        - Training results for every model.
        - Development results for every model.
        - Per-class precision, recall, and F1.
        - Micro-averaged precision, recall, and F1.
        - Macro-averaged precision, recall, and F1.
        - Overall token-level accuracy.

        Strongly encouraged:

        - Confusion matrix.
        - Learning curves for BiLSTM.
        - A compact comparison table with HMM vs BiLSTM.

In [26]:
def metrics_row(model, split, results):
    return {
        "Model": model,
        "Split": split,
        "Accuracy": results["accuracy"],
        "Micro Precision": results["micro"]["precision"],
        "Micro Recall": results["micro"]["recall"],
        "Micro F1": results["micro"]["f1"],
        "Macro Precision": results["macro"]["precision"],
        "Macro Recall": results["macro"]["recall"],
        "Macro F1": results["macro"]["f1"],
    }

comparison_rows = []
if "train_hmm_results" in globals() and "dev_hmm_results" in globals():
    comparison_rows.extend([
        metrics_row("HMM", "Train", train_hmm_results),
        metrics_row("HMM", "Dev", dev_hmm_results),
    ])

if "train_bilstm_results" in globals() and "dev_bilstm_results" in globals():
    comparison_rows.extend([
        metrics_row("BiLSTM", "Train", train_bilstm_results),
        metrics_row("BiLSTM", "Dev", dev_bilstm_results),
    ])

try:
    import pandas as pd
    results_table = pd.DataFrame(comparison_rows)
    display(results_table.style.format({
        "Accuracy": "{:.4f}",
        "Micro Precision": "{:.4f}",
        "Micro Recall": "{:.4f}",
        "Micro F1": "{:.4f}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}",
        "Macro F1": "{:.4f}",
    }))
except ImportError:
    for row in comparison_rows:
        print(row)


,Model,Split,Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1
0,HMM,Train,0.9599,0.9599,0.9599,0.9599,0.8830,0.8847,0.8798
1,HMM,Dev,0.9551,0.9551,0.9551,0.9551,0.8776,0.8838,0.8769


In [27]:
# TODO: Plot learning curves or confusion matrix if available.

## 8. Analysis & Conclusion

        Discuss the results in your own words.

        Suggested analysis questions:

        - Which model performed better on the development set?
        - Is there a gap between training and development performance?
        - Which POS tags were easiest?
        - Which POS tags were hardest?
        - Is macro F1 lower than micro F1? If yes, explain how rare classes affect this.
        - What kinds of errors happen often?
        - What are the limitations of your approach?

        Conclude with a short summary of what you learned and what could be improved in future work.

In [28]:
# TODO: Optional error analysis.
        # Print examples where the model made mistakes.

## 9. AI Usage Statement

        Write the required AI usage disclosure here.

        Example wording to edit:

        > We used AI tools for debugging, organization suggestions, and feedback on notebook structure. The core modeling decisions, experiment execution, result verification, and final analysis were reviewed and completed by the team. We are responsible for every claim, result, and line of code in this submission.

        Also mention which parts were written entirely by the team without AI assistance.

## 10. References

        Add all references used in the project.

        Suggested references:

        - Toutanova, K., Klein, D., Manning, C. D., & Singer, Y. (2003). Feature-Rich Part-of-Speech Tagging with a Cyclic Dependency Network. HLT-NAACL.
        - Huang, Z., Xu, W., & Yu, K. (2015). Bidirectional LSTM-CRF Models for Sequence Tagging. arXiv:1508.01991.
        - PyTorch documentation.
        - English Penn Treebank / CoNLL dataset source provided by the course instructor.